In [0]:
# CELL 1: Install & Verify Libraries
# -------------------------------------------------------------------------
# 1. Install the libraries
%pip install beautifulsoup4 gdown
 
# 2. Verify installation immediately
try:
    from bs4 import BeautifulSoup
    import gdown
    print("✅ BeautifulSoup4 and gdown are installed and working correctly.")
except ImportError as e:
    print(f"❌ Error: {e}. Please re-run this cell.")

In [0]:
# CELL 2: The Master Ingestion Function
# -------------------------------------------------------------------------
import gdown
import requests
import re
import os
import tempfile
import shutil
from datetime import datetime, timedelta
from azure.storage.filedatalake import DataLakeServiceClient
 
# -------------------------------------------------------------------------
# 1. Azure Storage Configuration
# -------------------------------------------------------------------------
storage_account_name = "databrickslabjp"
storage_account_key = "MOVED_FOR_SECURITY"
 
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)
spark.conf.set("fs.azure.createRemoteFileSystemDuringInitialization", "true")
 
# Google Drive folder containing the data files
GDRIVE_FOLDER_ID = "1dt-p1AH_8tQo9MHHvteXH3JloQCwP_f9"
 
# -------------------------------------------------------------------------
# 2. Helper: Resolve file name → Google Drive file ID
# -------------------------------------------------------------------------
def _resolve_file_id(folder_id, target_filename):
    """Find a file's Google Drive ID by name from a public folder page.
 
    Parses the folder HTML to locate the file without downloading everything.
    Returns None if not found (caller falls back to full folder download).
    """
    try:
        url = f"https://drive.google.com/drive/folders/{folder_id}"
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        resp = requests.get(url, headers=headers, timeout=30)
        text = resp.text
 
        # Google Drive embeds file metadata in the page HTML.
        # Search for the target filename and extract nearby file IDs.
        pos = text.lower().find(target_filename.lower())
        if pos == -1:
            return None
 
        # Extract file ID candidates from a window around the filename
        window = text[max(0, pos - 300):pos + len(target_filename) + 300]
        candidates = re.findall(r'"([a-zA-Z0-9_-]{25,45})"', window)
 
        # Filter out unlikely candidates (HTML attributes, CSS classes, etc.)
        skip_prefixes = ('data-', 'aria-', 'class-', 'style', 'http')
        for c in candidates:
            if c.lower() == target_filename.lower().replace('.csv', ''):
                continue
            if any(c.lower().startswith(p) for p in skip_prefixes):
                continue
            return c
    except Exception:
        pass
 
    return None
 
# -------------------------------------------------------------------------
# 3. The Reusable Ingestion Function
# -------------------------------------------------------------------------
def ingest_dataset(dataset_name, file_pattern):
    """Download a file by name from Google Drive and ingest to ADLS bronze layer.
 
    1. Creates bronze/{dataset_name}/landing/ and archive/ directories
    2. Archives existing file (renames with current date, moves to archive)
    3. Finds the file in Google Drive folder by name
    4. Downloads and stream-uploads to ADLS landing zone
    """
    print(f"\n🚀 STARTING INGESTION: {dataset_name.upper()}")
    print("=" * 50)
 
    # Define ADLS paths
    base_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/{dataset_name}"
    landing_zone = f"{base_path}/landing/"
    archive_zone = f"{base_path}/archive/"
 
    # A. Create directory structure
    dbutils.fs.mkdirs(landing_zone)
    dbutils.fs.mkdirs(archive_zone)
    print(f"   ✅ Directories verified: {landing_zone}")
 
    # B. Archive existing file if present
    current_file = f"{landing_zone}{file_pattern}"
    try:
        dbutils.fs.ls(current_file)
        today = datetime.now().strftime("%Y-%m-%d")
        archive_path = f"{archive_zone}{today}.csv"
        dbutils.fs.mv(current_file, archive_path)
        print(f"   📦 Archived existing file → {archive_path}")
    except Exception:
        print(f"   ℹ️ No existing '{file_pattern}' to archive. First run.")
 
    # C. Find and download file from Google Drive by name
    tmp_dir = tempfile.mkdtemp()
    try:
        local_path = os.path.join(tmp_dir, file_pattern)
 
        # Try to resolve file ID by name (avoids downloading entire folder)
        file_id = _resolve_file_id(GDRIVE_FOLDER_ID, file_pattern)
 
        if file_id:
            print(f"   🔍 Found '{file_pattern}' in folder (ID: {file_id[:8]}...)")
            gdown.download(id=file_id, output=local_path, quiet=False)
        else:
            # Fallback: download entire folder and locate file by name
            print(f"   🔍 Name lookup failed, downloading full folder...")
            gdown.download_folder(
                f"https://drive.google.com/drive/folders/{GDRIVE_FOLDER_ID}",
                output=tmp_dir, quiet=False
            )
            # Find the file in downloaded contents
            for root, dirs, files in os.walk(tmp_dir):
                for f in files:
                    if f.lower() == file_pattern.lower():
                        local_path = os.path.join(root, f)
                        break
 
        if not os.path.exists(local_path):
            raise FileNotFoundError(f"❌ '{file_pattern}' not found in Google Drive folder.")
 
        file_size = os.path.getsize(local_path)
        print(f"   ✅ Downloaded: {file_pattern} ({file_size:,} bytes)")
 
        # D. Stream-upload to ADLS using Azure SDK (memory-efficient)
        service = DataLakeServiceClient(
            account_url=f"https://{storage_account_name}.dfs.core.windows.net",
            credential=storage_account_key
        )
        fs_client = service.get_file_system_client("bronze")
        file_client = fs_client.get_file_client(f"{dataset_name}/landing/{file_pattern}")
 
        print(f"   ⬆️  Uploading to ADLS...")
        with open(local_path, "rb") as data:
            file_client.upload_data(data, overwrite=True)
 
        print(f"   ✅ Uploaded to: {current_file}")
 
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)
 
    print(f"✅ FINISHED: {dataset_name} is ready.\n")
 
print("Master Ingestion Function Loaded.")

In [0]:
# CELL 3: Execute Ingestion
# -------------------------------------------------------------------------
 
# 1. Ingest Units (Product Dimension)
ingest_dataset(
    dataset_name="units",
    file_pattern="units.csv"
)
 
# 2. Ingest Projects (Status Dimension)
ingest_dataset(
    dataset_name="projects",
    file_pattern="projects.csv"
)